In [0]:
# ============================================================
# Silver — Source 13: S3 Lambda Image Metadata
#
# Transformations:
#   - Cast event_time to timestamp
#   - Normalise image_format, colour_space, image_position
#   - Validate width_px and height_px > 0
#   - Reject null event_id or product_sku → quarantine
#   - Deduplicate on event_id
#
# Source:  bronze.src_13_images.image_metadata
# Target:  silver.src_13_images.image_metadata
# Quarantine: silver.quarantine.src_13_images
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_13_images.image_metadata'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_13_images'

VALID_FORMATS = ['jpg', 'jpeg', 'png', 'webp', 'gif', 'tiff']
VALID_COLOUR_SPACES = ['RGB', 'CMYK', 'sRGB', 'LAB']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_13_images')
print('Silver Source 13 Image Metadata — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_13_images.image_metadata')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamp
df = bronze.withColumn('event_time', F.to_timestamp(F.col('event_time')))

# Normalise
df = df \
    .withColumn('image_format',   F.lower(F.trim(F.col('image_format')))) \
    .withColumn('product_sku',    F.upper(F.trim(F.col('product_sku')))) \
    .withColumn('image_position', F.lower(F.trim(F.col('image_position'))))

# Bad rows
bad = df.filter(
    F.col('event_id').isNull() |
    F.col('product_sku').isNull() |
    F.col('event_time').isNull() |
    F.col('width_px').isNull() |
    F.col('height_px').isNull() |
    (F.col('width_px') <= 0) |
    (F.col('height_px') <= 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('image_metadata'))

# Good rows
good = df.filter(
    F.col('event_id').isNotNull() &
    F.col('product_sku').isNotNull() &
    F.col('event_time').isNotNull() &
    F.col('width_px').isNotNull() &
    F.col('height_px').isNotNull() &
    (F.col('width_px') > 0) &
    (F.col('height_px') > 0)
).dropDuplicates(['event_id'])

bad_count = bad.count()
good_count = good.count()
print(f'Images: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Image quality stats
print('\nImage quality breakdown:')
good.agg(
    F.sum(F.col('is_valid_format').cast('int')).alias('valid_format'),
    F.sum(F.col('meets_min_size').cast('int')).alias('meets_min_size'),
    F.sum(F.col('has_white_background').cast('int')).alias('has_white_bg'),
    F.count('*').alias('total')
).show()

# Write
if spark.catalog.tableExists(TARGET_TABLE):
    from delta.tables import DeltaTable
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.event_id = s.event_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('✅ Written')

# Quarantine
if bad_count > 0:
    bad.select(
        F.lit('src_13_images').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_13_images.image_metadata: {count} rows')
spark.sql(f"""
    SELECT image_format, colour_space, COUNT(*) as cnt
    FROM {TARGET_TABLE}
    GROUP BY image_format, colour_space
    ORDER BY cnt DESC
""").show()
